# 05 · RAG B: evidencias de Wikipedia (dev)

> **Ejecutar en Kaggle:** GPU T4 ×2, Internet activado (consulta Wikipedia), secreto `HF_TOKEN`. La recuperación se puede hacer en local (~10–15 min en CPU).
> **Salidas:** `results/rag/evidencias_dev.jsonl` y `results/llm/ragB_dev_*.csv`.

Para cada noticia de dev se buscan evidencias en Wikipedia en español (`WikipediaSource`, en `src/rag/evidence.py`) y se le dan al modelo antes del texto.

- La consulta sale del titular o la primera frase; `Link` solo sirve para excluir el dominio original.
- **Fecha:** el corpus no la trae; se usa la cota 2018-12-31 y la revisión de Wikipedia anterior a ella. [CONFIRMAR con el equipo.]
- **Sin evidencia** es un caso propio: el prompt aclara que no significa «falsa».
- **Control:** el mismo prompt sin evidencias (`NullSource`), para separar el efecto del prompt del de las evidencias.

Test no se usa.

In [1]:
import os, sys, subprocess, shutil
from pathlib import Path

IN_KAGGLE = Path("/kaggle").exists()
# Rama que contiene el código nuevo (src/ y notebooks 05-06). Cambiar si se sube a otra.
REPO, BRANCH = "https://github.com/Rodrigo-Romay/Practica-TAPLN.git", "feature/RAG"
PAQUETES = ['transformers', 'accelerate', 'bitsandbytes']

if IN_KAGGLE:
    os.environ["HF_HOME"] = "/tmp/hf"   # más disco libre que /kaggle/working
    ROOT = Path("/kaggle/working/Practica-TAPLN")

    def secreto(nombre):
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(nombre)
        except Exception:
            return None

    if not ROOT.exists():
        gh = secreto("GH_TOKEN")   # solo hace falta si el repo es privado
        url = REPO.replace("https://", f"https://{gh}@") if gh else REPO
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, str(ROOT)], check=True)
    if PAQUETES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *PAQUETES], check=True)
    hf = secreto("HF_TOKEN")
    if hf:
        os.environ["HF_TOKEN"] = hf
    else:
        print("Sin HF_TOKEN: solo funcionarán los modelos no restringidos.")
    # Resultados intermedios no versionados (opcional): dataset privado `fakedes-resultados`
    extra = Path("/kaggle/input/fakedes-resultados/results")
    if extra.exists():
        shutil.copytree(extra, ROOT / "results", dirs_exist_ok=True)
else:
    ROOT = Path.cwd().resolve().parent

sys.path.insert(0, str(ROOT / "src"))
if not (ROOT / "data" / "processed" / "train.csv").exists():
    raw = ROOT / "data" / "raw"
    subprocess.run([sys.executable, str(ROOT / "src" / "process_data.py"),
                    "--train", str(raw / "train.xlsx"), "--dev", str(raw / "development.xlsx"),
                    "--test", str(raw / "test.xlsx"), "--out", str(ROOT / "data" / "processed")], check=True)

Sin HF_TOKEN: solo funcionarán los modelos no restringidos.
Duplicados train/dev eliminados de dev: 4
train: 676 filas | Fake=338 True=338 | titulares vacíos=0 | duplicados internos=0
dev: 291 filas | Fake=140 True=151 | titulares vacíos=0 | duplicados internos=1
test: 572 filas | Fake=286 True=286 | titulares vacíos=72 | duplicados internos=0
Guardado en /kaggle/working/Practica-TAPLN/data/processed


In [2]:
import numpy as np
import pandas as pd
import torch

import data_utils as du
import preprocessing as pp
import evaluation as ev
import experiments as ex
import llm
import llm_fewshot as fs

splits = du.load_clean_splits(ROOT)
media_re = pp.media_regex_from_splits(splits)   # nombres de medio de train/dev
HAY_GPU = torch.cuda.is_available()
MAX_TOKENS = 4000   # si un prompt no cabe, se quitan ejemplos (queda en n_ejemplos); la noticia no se recorta
print({k: len(v) for k, v in splits.items()}, "| GPU:", HAY_GPU)

{'train': 676, 'dev': 290, 'test': 572} | GPU: True


## Recuperación de evidencias (CPU + Internet, reanudable)

In [3]:
from rag import evidence as evd

RAG = ROOT / "results" / "rag"
ruta_ev = RAG / "evidencias_dev.jsonl"
K_EV = 3
dev_in = ex.inputs_for(splits, media_re, "dev")
evidencias = evd.retrieve_all(dev_in, splits["dev"]["Link"], evd.WikipediaSource(), ruta_ev,
                              k=K_EV, fecha_max=ex.FECHA_MAX["dev"])
t = evd.evidence_table(ruta_ev)
print(f"noticias: {len(t)} | sin evidencia: {(t.n == 0).mean():.1%} | con error de red: {(t.error != '').sum()}")
print("Si hay errores de red: borrar esas líneas del JSONL y volver a ejecutar la celda.")
t.n.value_counts().sort_index()

noticias: 290 | sin evidencia: 46.6% | con error de red: 0
Si hay errores de red: borrar esas líneas del JSONL y volver a ejecutar la celda.


n
0    135
1     46
2     41
3     68
Name: count, dtype: int64

### Revisión manual

Muestra de 25 noticias de dev; 13 tienen evidencia y se juzgan por los títulos devueltos.

| Criterio | Casos (de 13) |
|---|---|
| Trata el mismo hecho | 0 |
| Misma entidad o tema, sin el hecho (Big Show, paro UNAM, Blue Demon, Messi) | 4 |
| Ruido (p. ej. «2025 \| 2006 \| 2001», Nayib Bukele) | 9 |

Muestra pequeña, pero el patrón es claro: Wikipedia no recoge noticias locales ni bulos concretos y la búsqueda devuelve coincidencias de palabras sueltas.

In [ ]:
pd.set_option("display.max_colwidth", 200)
t[t.n > 0].sample(25, random_state=du.SEED)[["Id", "query", "n", "titulos"]]


## Experimentos

> **No ejecutados:** RAG B se descarta tras la revisión manual. Las salidas de la primera ejecución en Kaggle se han eliminado porque la recuperación falló (286/290 errores de red) y no medían RAG. Las celdas se conservan por si se cambia la fuente.

In [ ]:
# Referencia a batir: la configuración elegida en 03 para cada modelo (few-shot fijo, k=8)
PROMPT = {"gemma4-31b": "base", "qwen38-27b": "detallado"}
REF = {"gemma4-31b": "ajuste_dev_gemma4-31b__base_k8.csv", "qwen38-27b": "ajuste_dev_qwen38-27b__detallado_k8.csv"}
EXPS_B = [
    ex.experiment(splits, media_re, "dev", modo="rag_b", k=K_EV, evidencias=evidencias, fuente="wikipedia"),
    ex.experiment(splits, media_re, "dev", modo="rag_b", k=K_EV, evidencias={}, fuente="ninguna"),   # control
]
EXPS = {clave: EXPS_B for clave in ex.FINALISTAS}
for e in EXPS_B:
    print(e["experimento"], e["pid"])
con = next((i for i, r in enumerate(dev_in.itertuples(index=False)) if evidencias.get(r.Id)), 0)
print(ex.show_prompt(EXPS_B[0], con))


## Ejecución (GPU)

In [ ]:
MODELOS = list(ex.FINALISTAS)   # gemma4-31b (principal) y qwen38-27b (contraste)
if HAY_GPU:
    for clave in MODELOS:
        ex.run_all(ROOT, EXPS[clave], [clave], borrar_cache=IN_KAGGLE)
else:
    print("Sin GPU: no se ejecutan modelos. Con los CSV ya descargados en results/llm/ se puede ver el resumen.")

## Resultados en dev

In [ ]:
COLS = ["modelo", "decision", "f1_fake", "f1_fake_inf", "f1_fake_sup", "auc", "auc_inf", "auc_sup",
        "precision", "recall", "pct_pred_fake", "s_noticia", "tokens", "completo"]
tabla = ex.dev_table(ROOT, "dev")
if len(tabla):
    tabla = tabla.sort_values("auc", ascending=False)
    tabla.to_csv(ROOT / "results" / "llm" / "ragB_dev_resumen.csv", index=False)
    display(tabla[COLS].round(3).reset_index(drop=True))

In [ ]:
# Comparación pareada (bootstrap) de cada experimento nuevo frente a la configuración elegida en 03
# (few-shot fijo k=8) y frente al zero-shot del piloto, por modelo
filas = []
for clave in MODELOS:
    sistemas = {}
    for nombre, f in [("few-shot k=8 (03)", REF[clave]), ("zero-shot (piloto)", f"piloto_dev_{clave}.csv")]:
        if (ROOT / "results" / "llm" / f).exists():
            sistemas[nombre] = ex.to_predictions(pd.read_csv(ROOT / "results" / "llm" / f), nombre)
    for e in EXPS[clave]:
        p = ex.out_path(ROOT, e, clave)
        if fs.is_complete(p, len(e["inputs"])):
            sistemas[f"{e['experimento']} {e['pid']}"] = ex.to_predictions(pd.read_csv(p), e["pid"])
    nuevos = [s for s in sistemas if s not in ("few-shot k=8 (03)", "zero-shot (piloto)")]
    if nuevos and len(sistemas) > 1:
        c = ev.compare_predictions(ROOT, sistemas, "dev")
        filas.append(c[c["A"].isin(nuevos) ^ c["B"].isin(nuevos)].assign(modelo=clave))
if filas:
    display(pd.concat(filas, ignore_index=True).round(3))


In [ ]:
# ¿Ayuda tener evidencia? Métricas por separado para noticias con y sin evidencia (descriptivo)
for clave in MODELOS:
    p = ex.out_path(ROOT, EXPS_B[0], clave)
    if fs.is_complete(p, len(dev_in)):
        r = pd.read_csv(p)
        for nombre, g in r.groupby(r.n_evidencias > 0):
            print(clave, "con evidencia" if nombre else "sin evidencia",
                  ev.evaluate(g.label, ev.decide_threshold(g.logit_diff), g.logit_diff))

## Lectura de resultados y decisión

- **Recuperación (local, 0 errores de red):** 53,4 % de las 290 noticias con 1–3 artículos y 46,6 % sin evidencia.
- **Relevancia:** 0/13 tratan el mismo hecho, 4/13 la misma entidad y 9/13 son ruido. La cobertura del 53 % es aparente.
- **Motivo:** Wikipedia no cubre este tipo de noticias; en el mejor caso el modelo ignora las evidencias y en el peor se distrae, a cambio de ~3 h de GPU.
- Los LLM con evidencias no se han ejecutado, así que no hay comparación con 03.

**Decisión:** RAG B con Wikipedia se descarta. Si se retoma, haría falta otra fuente (verificadores, hemeroteca) o recuperar por entidades; la interfaz `EvidenceSource` se puede reutilizar. Test no se usa.